In [184]:
!pip install groq gtts

In [185]:
import os
os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"

In [186]:
import os
import base64
from groq import Groq
from gtts import gTTS
from IPython.display import Audio, display, HTML
from google.colab import output

In [187]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [188]:
# candidate_info is automatically extracted from the uploaded resume
candidate_info = {}

In [189]:
!pip install pymupdf

In [190]:
import fitz

def parse_resume(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    return full_text

In [191]:
def extract_candidate_info(pdf_path):
    raw_text = parse_resume(pdf_path)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""Extract the following information from this resume and return it as a Python dictionary with exactly these keys:
- name
- skills (as a list)
- projects (as a list of project names)
- experience (fresher / 1-2 years / 3-5 years etc.)

Resume text:
{raw_text}

Return ONLY the dictionary, nothing else. Example format:
{{
    "name": "John",
    "skills": ["Python", "Machine Learning"],
    "projects": ["Sentiment Analyzer", "Image Classifier"],
    "experience": "Fresher"
}}"""
        }]
    )

    import ast
    result = response.choices[0].message.content.strip()
    candidate_info = ast.literal_eval(result)
    return candidate_info

In [192]:
from google.colab import files

print("Upload your resume PDF...")
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
candidate_info = extract_candidate_info(pdf_filename)

print("Extracted info:")
print(candidate_info)

Upload your resume PDF...


Saving ML Resume.pdf to ML Resume (5).pdf
Extracted info:
{'name': 'Venkat Ramanan', 'skills': ['Python', 'Machine Learning', 'Deep Learning', 'TensorFlow', 'Keras', 'Scikit-learn', 'NLTK', 'OpenCV', 'Pandas', 'NumPy', 'Git'], 'projects': ['Sentiment Analyzer using LSTM', 'Image Classifier using CNN', 'Stock Price Predictor'], 'experience': 'Fresher'}


In [193]:
conversation = [
    {
        "role": "system",
        "content": f"""You are Rachel, a professional senior technical interviewer at a top tech company.
You are interviewing {candidate_info['name']} for a role in {', '.join(candidate_info['skills'])}.
Their projects include: {', '.join(candidate_info['projects'])}.
They are a {candidate_info['experience']}.

Your job is to conduct a thorough 20-30 minute interview. Cover ALL of these areas:

1. TECHNICAL DEPTH — Ask about their projects in detail. Go deep, ask follow-ups.
2. FUNDAMENTALS — Ask core concepts related to their skills (ML theory, Python concepts etc.)
3. PROBLEM SOLVING — Give them a small scenario or problem to think through.
4. BEHAVIORAL — Ask about teamwork, challenges faced, how they handle failure.
5. CAREER INTENT — Why this field, where they see themselves, what they want to learn.

Rules:
- Ask ONE question at a time
- Always ask a follow-up if the answer is vague or shallow
- Vary between all 5 areas above, don't stick to just projects
- Ask at least 15-20 questions total
- Only say exactly [INTERVIEW COMPLETE] after covering all 5 areas thoroughly
- Be encouraging but push for depth
- If they give a one line answer, ask them to elaborate"""
    }
]

In [194]:
def speak(text):
    tts = gTTS(text=text, lang='en')
    tts.save("question.mp3")
    display(Audio("question.mp3", autoplay=True))

In [195]:
audio_b64_storage = []

def get_audio(b64):
    audio_b64_storage.clear()
    audio_b64_storage.append(b64)

output.register_callback('notebook.get_audio', get_audio)

In [196]:
record_button_html = """
<button id="startBtn" style="padding:12px 24px;font-size:16px;cursor:pointer;background:green;color:white;border:none;border-radius:8px">
  🎤 Start Recording
</button>
<button id="stopBtn" style="padding:12px 24px;font-size:16px;cursor:pointer;background:red;color:white;border:none;border-radius:8px;display:none">
  ⏹ Stop Recording
</button>
<p id="status" style="font-size:14px;margin-top:10px">Click Start and speak your answer</p>
<script>
let recorder, chunks = [], stream;
document.getElementById('startBtn').onclick = async () => {
    stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    recorder = new MediaRecorder(stream);
    chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.start();
    document.getElementById('startBtn').style.display = 'none';
    document.getElementById('stopBtn').style.display = 'inline';
    document.getElementById('status').innerText = '🔴 Recording... speak now';
};
document.getElementById('stopBtn').onclick = () => {
    recorder.stop();
    stream.getTracks().forEach(t => t.stop());
    recorder.onstop = () => {
        const blob = new Blob(chunks, {type: 'audio/webm'});
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
            const b64 = reader.result.split(',')[1];
            google.colab.kernel.invokeFunction('notebook.get_audio', [b64], {});
            document.getElementById('status').innerText = '✅ Done! Now run the next cell.';
            document.getElementById('stopBtn').style.display = 'none';
        };
    };
};
</script>
"""

In [197]:
def show_record_button():
    display(HTML(record_button_html))

In [198]:
def transcribe():
    audio_bytes = base64.b64decode(audio_b64_storage[0])
    with open("voice_answer.webm", "wb") as f:
        f.write(audio_bytes)
    with open("voice_answer.webm", "rb") as f:
        result = client.audio.transcriptions.create(
            model="whisper-large-v3",
            file=("voice_answer.webm", f),
        )
    return result.text

In [199]:
def ask_ai(user_message):
    conversation.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=conversation
    )
    ai_reply = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": ai_reply})
    return ai_reply

In [200]:
def generate_report():
    full_transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in conversation
        if m['role'] != 'system'
    )
    report_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""Based on this interview transcript, generate a detailed candidate report.

{full_transcript}

Give scores out of 10 for:
1. Technical Knowledge
2. Communication Clarity
3. Problem Solving
4. Project Depth
5. Confidence

Also write:
- A 3 sentence overall summary
- 3 strengths
- 3 areas to improve
- Hiring recommendation (Strong Yes / Yes / Maybe / No)

Format it clearly."""
        }]
    )
    return report_response.choices[0].message.content

In [201]:
def next_turn():
    print("Transcribing your answer...")
    user_answer = transcribe()
    print(f"\nYOU: {user_answer}\n")
    next_question = ask_ai(user_answer)
    if "[INTERVIEW COMPLETE]" in next_question:
        print("RACHEL: Thank you! Generating your report...\n")
        speak("Thank you so much! Generating your report now.")
        print("\n" + "=" * 50)
        print("           CANDIDATE REPORT")
        print("=" * 50)
        print(generate_full_report(conversation))
        return
    print(f"RACHEL: {next_question}\n")
    speak(next_question)
    print("--- Record your next answer ---")
    show_record_button()

In [202]:
!pip install fastapi uvicorn nest-asyncio pyngrok

In [203]:
import nest_asyncio
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

In [204]:
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

In [205]:
@app.post("/start-interview")
async def start_interview(file: UploadFile = File(...)):
    # Save uploaded resume
    contents = await file.read()
    with open("resume.pdf", "wb") as f:
        f.write(contents)

    # Extract candidate info from resume
    candidate = extract_candidate_info("resume.pdf")

    # Build conversation
    conversation = [
        {
            "role": "system",
            "content": f"""You are Rachel, a professional senior technical interviewer at a top tech company.
You are interviewing {candidate['name']} for a role in {', '.join(candidate['skills'])}.
Their projects include: {', '.join(candidate['projects'])}.
They are a {candidate['experience']}.

Your job is to conduct a thorough 20-30 minute interview. Cover ALL of these areas:

1. TECHNICAL DEPTH — Ask about their projects in detail. Go deep, ask follow-ups.
2. FUNDAMENTALS — Ask core concepts related to their skills (ML theory, Python concepts etc.)
3. PROBLEM SOLVING — Give them a small scenario or problem to think through.
4. BEHAVIORAL — Ask about teamwork, challenges faced, how they handle failure.
5. CAREER INTENT — Why this field, where they see themselves, what they want to learn.

Rules:
- Ask ONE question at a time
- Always ask a follow-up if the answer is vague or shallow
- Vary between all 5 areas above, don't stick to just projects
- Ask at least 15-20 questions total
- Only say exactly [INTERVIEW COMPLETE] after covering all 5 areas thoroughly
- Be encouraging but push for depth
- If they give a one line answer, ask them to elaborate"""
        }
    ]

    # Get first question
    first_question = ask_ai(conversation, "Hello, I am ready for my interview.")

    return {
        "candidate": candidate,
        "conversation": conversation,
        "question": first_question
    }

In [206]:
@app.post("/next-question")
async def next_question(data: dict):
    conversation = data["conversation"]
    user_answer = data["answer"]

    # Add user answer to conversation
    conversation.append({"role": "user", "content": user_answer})

    # Get AI response
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=conversation
    )

    ai_reply = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": ai_reply})

    # Check if interview is complete
    if "[INTERVIEW COMPLETE]" in ai_reply:
        report = generate_report(conversation)
        return {
            "done": True,
            "conversation": conversation,
            "report": report
        }

    return {
        "done": False,
        "conversation": conversation,
        "question": ai_reply
    }

In [207]:
def generate_report(conversation):
    full_transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in conversation
        if m['role'] != 'system'
    )
    report_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""Based on this interview transcript, generate a detailed candidate report.

{full_transcript}

Give scores out of 10 for:
1. Technical Knowledge
2. Communication Clarity
3. Problem Solving
4. Project Depth
5. Confidence

Also write:
- A 3 sentence overall summary
- 3 strengths
- 3 areas to improve
- Hiring recommendation (Strong Yes / Yes / Maybe / No)

Format it clearly."""
        }]
    )
    return report_response.choices[0].message.content

In [208]:
ngrok.set_auth_token("your-ngrok-api-key-here")

In [209]:
def get_webcam_snapshot_html():
    return """
    <video id="video" width="320" height="240" autoplay style="border-radius:8px"></video>
    <canvas id="canvas" width="320" height="240" style="display:none"></canvas>
    <p id="cam-status" style="font-size:13px;color:gray">Camera starting...</p>

    <script>
    async function startCamera() {
        const video = document.getElementById('video');
        const stream = await navigator.mediaDevices.getUserMedia({ video: true });
        video.srcObject = stream;
        document.getElementById('cam-status').innerText = '🟢 Camera on — monitoring for cheating';

        // Take snapshot every 10 seconds
        setInterval(() => {
            const canvas = document.getElementById('canvas');
            const ctx = canvas.getContext('2d');
            ctx.drawImage(video, 0, 0, 320, 240);
            const b64 = canvas.toDataURL('image/jpeg').split(',')[1];
            google.colab.kernel.invokeFunction('notebook.analyze_frame', [b64], {});
        }, 10000);
    }

    startCamera();
    </script>
    """

In [210]:
import base64
import datetime

cheat_log = []

def analyze_frame(b64_image):
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{b64_image}"
                    }
                },
                {
                    "type": "text",
                    "text": """You are a cheating detection system for an online interview.
Analyze this webcam frame and check for:
1. Is the candidate looking away from screen repeatedly?
2. Is there another person visible?
3. Is the candidate's face not visible or out of frame?
4. Is the candidate looking down at notes?
5. Is another device (phone/tablet) visible?

Reply in this exact format:
SUSPICIOUS: Yes/No
REASON: (one line explanation)
SEVERITY: Low/Medium/High"""
                }
            ]
        }]
    )

    result = response.choices[0].message.content

    if "SUSPICIOUS: Yes" in result:
        timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        cheat_log.append(f"{timestamp} — {result}")
        print(f"⚠️ Suspicious activity at {timestamp}: {result}")
    else:
        print(f"✅ Frame clear")

output.register_callback('notebook.analyze_frame', analyze_frame)

In [211]:
def start_cheat_detection():
    print("🎥 Starting cheat detection camera...")
    display(HTML(get_webcam_snapshot_html()))

In [212]:
def get_cheat_summary():
    if not cheat_log:
        return "No suspicious activity detected. Integrity Score: 10/10"

    summary_response = client.chat.completions.create(
        model="eta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{
            "role": "user",
            "content": f"""Based on these cheating flags detected during the interview, give a brief integrity summary:

{chr(10).join(cheat_log)}

Write:
- Integrity Score: X/10
- Number of suspicious moments
- Brief summary of what was detected"""
        }]
    )
    return summary_response.choices[0].message.content

In [213]:
def generate_full_report(conversation):
    interview_report = generate_report(conversation)
    cheat_summary = get_cheat_summary()

    return f"""
{interview_report}

{'='*50}
INTEGRITY REPORT
{'='*50}
{cheat_summary}
"""

In [214]:
DID_API_KEY = "your-did-api-key-here"
AVATAR_IMAGE_URL = "https://clips-presenters.d-id.com/amy/image.png"

In [215]:
import requests
import time

def speak_with_avatar(text):
    # Create talking avatar video
    response = requests.post(
        "https://api.d-id.com/talks",
        headers={
            "Authorization": f"Basic {DID_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "source_url": AVATAR_IMAGE_URL,
            "script": {
                "type": "text",
                "input": text,
                "provider": {
                    "type": "microsoft",
                    "voice_id": "en-US-JennyNeural"
                }
            }
        }
    )

    talk_id = response.json().get("id")

    if not talk_id:
        print("D-ID failed, falling back to audio only")
        speak(text)
        return

    # Wait for video to be ready
    for _ in range(20):
        time.sleep(3)
        result = requests.get(
            f"https://api.d-id.com/talks/{talk_id}",
            headers={"Authorization": f"Basic {DID_API_KEY}"}
        )
        data = result.json()
        if data.get("status") == "done":
            video_url = data.get("result_url")
            display(HTML(f'''
                <video width="320" height="240" autoplay controls style="border-radius:12px">
                    <source src="{video_url}" type="video/mp4">
                </video>
            '''))
            return

    # If timeout, fall back to audio
    print("Timeout, falling back to audio")
    speak(text)

In [216]:
# Override speak function to use avatar
def speak(text):
    speak_with_avatar(text)

In [218]:
# Start cheat detection camera
start_cheat_detection()

🎥 Starting cheat detection camera...


✅ Frame clear
✅ Frame clear
✅ Frame clear
✅ Frame clear
✅ Frame clear
✅ Frame clear
✅ Frame clear
✅ Frame clear
✅ Frame clear
⚠️ Suspicious activity at 08:50:17: SUSPICIOUS: Yes
REASON: The candidate appears to be looking away from the screen and possibly using another device.
SEVERITY: Medium 

Reasoning: 
The candidate seems to be looking away from the screen, which could indicate they are reading from somewhere else or using another device. Their hand is covering part of their face, which might be obscuring their view or hiding something. However, without more context or frames, it's hard to determine the exact severity. 

To fully assess, I would need to analyze more frames or have access to more information about the interview setup and expectations. However, based on the provided image, these observations suggest potential suspicious activity. 

Here are the specific checks:
1. The candidate seems to be looking away from the screen.
2. No other person is visible.
3. The candidate

In [219]:
print("=" * 50)
print("   AUREETURE AI INTERVIEW SYSTEM")
print("=" * 50)
print(f"Candidate: {candidate_info['name']}")
print("=" * 50 + "\n")

first_question = ask_ai("Hello, I am ready for my interview.")
print(f"RACHEL: {first_question}\n")
speak(first_question)
print("--- Record your answer ---")
show_record_button()

   AUREETURE AI INTERVIEW SYSTEM
Candidate: Venkat Ramanan

RACHEL: I'm excited to conduct this interview with you, Venkat. Let's get started. I've reviewed your projects, and I'm impressed with the variety of topics you've worked on. I'd like to dive deeper into your Sentiment Analyzer using LSTM. Can you explain the architecture of your LSTM model, including the number of layers, the activation functions used, and how you handled the text preprocessing?

D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, falling back to audio only
D-ID failed, fallin

KeyboardInterrupt: 

In [220]:
next_turn()

Transcribing your answer...


IndexError: list index out of range

In [ ]:
import asyncio

ngrok_tunnel = ngrok.connect(8000)
print(f"Public URL: {ngrok_tunnel.public_url}")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

Public URL: https://literary-pauline-nonparty.ngrok-free.dev


INFO:     Started server process [183]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2401:4900:6272:caa4:181e:20ff:fee8:44be:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2401:4900:6272:caa4:181e:20ff:fee8:44be:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     115.96.213.252:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     115.96.213.252:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     115.96.213.252:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     115.96.213.252:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     36.255.16.60:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     36.255.16.60:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     36.255.16.60:0 - "POST /start-interview HTTP/1.1" 422 Unprocessable Entity
INFO:     36.255.16.60:0 - "POST /next-question HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

INFO:     36.255.16.60:0 - "POST /start-interview HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pymupdf/__init__.py", line 3004, in __init__
    doc = mupdf.fz_open_document(filename)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymupdf/mupdf.py", line 50789, in fz_open_document
    return _mupdf.fz_open_document(filename)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pymupdf.mupdf.FzErrorFormat: code=7: no objects found

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
         

INFO:     36.255.16.60:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     36.255.16.60:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     36.255.16.60:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     36.255.16.60:0 - "GET /openapi.json HTTP/1.1" 200 OK
